# 10 — Data Modeling

## 1. Why this concept exists

Data modeling decides **what a record means, how entities relate, and where facts live**.

A good model makes important questions easy to answer while preserving clear ownership of facts. A poor model creates duplication, ambiguity, update anomalies, and brittle analytics.

## 2. Short mental model

Start with four questions:

1. **What is the grain?** — What does one row represent?
2. **What are the entities?** — Customer, order, product, etc.
3. **What are the relationships?** — One-to-one, one-to-many, many-to-many.
4. **Which facts belong where?** — Store each fact at the grain where it is true.

We will model a small commerce system using SQLite.

The goal is not to memorize normalization rules. It is to learn to reason from business events to rows, keys, constraints and queries.

## Capability contract

**Capability:** C02 — Data Access Querying & Modeling  
**Workstream:** Revenue Data Platform  
**Primary roles:** Data Engineer|Analytics Engineer|Platform Engineer  
**You will prove:** Translate a business question into correct data access and defend the model/query behavior.


## 🖊️ Sketch note

![Sketch note — Data Modeling](../assets/sketch-notes/10-data-modeling.svg)

> **Visual-first rule:** spend 30–60 seconds explaining the sketch in your own words before reading the implementation. Then return to it after the experiment and redraw it from memory.


## Agent-native lens

**Agent can do:** routine implementation or a first-pass analysis.

**You must understand:** the mechanism and its assumptions.

**You must verify:** use tests, measurements or adversarial data that could prove the output wrong.

**You must decide:** the trade-off, failure response or production boundary.


## Apprenticeship bridge

Use this notebook to prepare for the following Acme Commerce work scenario(s). The scenario gives you symptoms and constraints; it does **not** give you the diagnosis.

- **S07 — Query plans and indexing (DATA-1102)** → [`../work_simulations/07_acme_commerce_latency_budget/README.md`](../work_simulations/07_acme_commerce_latency_budget/README.md)

**How to use the bridge:** finish the experiment and independent challenge here, then enter the ticket without reopening the notebook as an answer key. Bring your own prediction, evidence and verification plan.


## Measure

Record an observable result that lets you compare your prediction with what actually happened.


## Learning objectives

- state the grain of a table;
- choose keys and relationships;
- identify where duplication can enter;
- defend one modeling trade-off.


## Predict before you run

Predict the row count after each join based on the stated grain.


## Diagnose

If a result is wrong, trace it back to grain, key uniqueness or relationship assumptions before changing the query.


In [ ]:
import sqlite3
from pathlib import Path
import tempfile

db_path = Path(tempfile.gettempdir()) / "data_modeling_lab.db"
if db_path.exists():
    db_path.unlink()

con = sqlite3.connect(db_path)
con.execute("PRAGMA foreign_keys = ON")

con.executescript("""
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    country TEXT NOT NULL
);

CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    unit_price INTEGER NOT NULL CHECK(unit_price > 0)
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL REFERENCES customers(customer_id),
    order_date TEXT NOT NULL
);

CREATE TABLE order_items (
    order_id INTEGER NOT NULL REFERENCES orders(order_id),
    product_id INTEGER NOT NULL REFERENCES products(product_id),
    quantity INTEGER NOT NULL CHECK(quantity > 0),
    unit_price INTEGER NOT NULL CHECK(unit_price > 0),
    PRIMARY KEY (order_id, product_id)
);
""")

con.executemany("INSERT INTO customers VALUES (?, ?, ?)", [
    (1, "Asha", "IN"), (2, "Ben", "GB"), (3, "Chen", "SG")
])
con.executemany("INSERT INTO products VALUES (?, ?, ?)", [
    (101, "Notebook", 500), (102, "Pen", 100), (103, "Bag", 2000)
])
con.executemany("INSERT INTO orders VALUES (?, ?, ?)", [
    (1001, 1, "2026-01-05"), (1002, 1, "2026-01-08"), (1003, 2, "2026-01-09")
])
con.executemany("INSERT INTO order_items VALUES (?, ?, ?, ?)", [
    (1001, 101, 2, 500), (1001, 102, 3, 100),
    (1002, 103, 1, 2000), (1003, 101, 1, 500)
])
con.commit()

print("Tables:", [r[0] for r in con.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
)])

## 3. Establish the grain

Write the grain in plain English before writing SQL:

- `customers`: one row per customer.
- `products`: one row per product.
- `orders`: one row per order.
- `order_items`: one row per **product within an order**.

That last statement is especially important. It tells us why `(order_id, product_id)` can be the primary key and why quantity belongs in `order_items`.

In [ ]:
grain_checks = {
    "customers": "SELECT COUNT(*) = COUNT(DISTINCT customer_id) FROM customers",
    "products": "SELECT COUNT(*) = COUNT(DISTINCT product_id) FROM products",
    "orders": "SELECT COUNT(*) = COUNT(DISTINCT order_id) FROM orders",
    "order_items": "SELECT COUNT(*) = COUNT(DISTINCT order_id || ':' || product_id) FROM order_items",
}
for table, sql in grain_checks.items():
    ok = bool(con.execute(sql).fetchone()[0])
    print(table, "grain unique:", ok)
    assert ok

## 4. Relationships become joins

The model encodes:

```text
customer 1 ────< order 1 ────< order_item >──── 1 product
```

This is a **one-to-many** chain.

If you join a customer to order items, one customer can produce multiple rows. That is not duplication by itself; it is the expected consequence of the grain.

In [ ]:
sql = """
SELECT
    c.customer_id,
    c.name,
    o.order_id,
    oi.product_id,
    oi.quantity,
    oi.quantity * oi.unit_price AS line_value
FROM customers c
JOIN orders o ON o.customer_id = c.customer_id
JOIN order_items oi ON oi.order_id = o.order_id
ORDER BY c.customer_id, o.order_id, oi.product_id
"""
for row in con.execute(sql):
    print(row)

## 5. Model facts at the right grain

Notice that `order_items.unit_price` is stored separately from `products.unit_price`.

Why? Product catalog price can change later. An order needs the **price actually charged at the time of the order**.

This is a useful modeling principle:

> A historical event should preserve the facts needed to explain that event, even when a current reference value can change.

In [ ]:
con.execute("UPDATE products SET unit_price = 600 WHERE product_id = 101")
current_price = con.execute("SELECT unit_price FROM products WHERE product_id = 101").fetchone()[0]
historical_price = con.execute(
    "SELECT unit_price FROM order_items WHERE order_id = 1001 AND product_id = 101"
).fetchone()[0]

print("Current catalog price:", current_price)
print("Historical order price:", historical_price)
assert current_price == 600
assert historical_price == 500

## 6. Break it deliberately

Try these experiments:

1. Insert an `order_item` for a non-existent order.
2. Insert quantity `0`.
3. Duplicate an `(order_id, product_id)` pair.
4. Delete a customer that still has orders.
5. Move `quantity` into `products` and explain why the meaning becomes ambiguous.

The first four should be rejected by constraints. The fifth is a modeling thought experiment: a fact should live where its grain makes sense.

In [ ]:
experiments = [
    ("missing foreign key", "INSERT INTO order_items VALUES (9999, 101, 1, 500)"),
    ("invalid quantity", "INSERT INTO order_items VALUES (1001, 101, 0, 500)"),
    ("duplicate composite key", "INSERT INTO order_items VALUES (1001, 101, 1, 500)"),
]
for label, sql in experiments:
    try:
        con.execute(sql)
        con.commit()
        print(label, "→ unexpected success")
    except sqlite3.IntegrityError as exc:
        con.rollback()
        print(label, "→ rejected:", exc)

## 7. Measure the model with a business question

A model is useful when it supports correct questions.

Calculate revenue by customer from the event-level facts, then compare it with the number of orders. This also reinforces the difference between **row grain** and **aggregation grain**.

In [ ]:
revenue_sql = """
SELECT
    c.customer_id,
    c.name,
    COUNT(DISTINCT o.order_id) AS orders,
    SUM(oi.quantity * oi.unit_price) AS revenue
FROM customers c
JOIN orders o ON o.customer_id = c.customer_id
JOIN order_items oi ON oi.order_id = o.order_id
GROUP BY c.customer_id, c.name
ORDER BY c.customer_id
"""
result = con.execute(revenue_sql).fetchall()
print(result)
assert result == [(1, "Asha", 2, 3300), (2, "Ben", 1, 500)]

## 8. Production implications

Real data models usually balance several forces:

- normalized operational models reduce update anomalies;
- analytical models may deliberately denormalize for access patterns;
- surrogate and natural keys have different trade-offs;
- slowly changing reference data needs explicit history strategy;
- constraints protect assumptions close to the data;
- schema evolution must be treated as an operational concern;
- the model should be documented in terms of **grain and ownership of facts**.

**Engineering challenge:** extend the model with `payments` and decide its grain. Then answer: can one order have multiple payments? If yes, what should the primary key and relationship be?

In [ ]:
assert con.execute("PRAGMA foreign_keys").fetchone()[0] == 1
assert con.execute("SELECT COUNT(*) FROM customers").fetchone()[0] == 3
assert con.execute("SELECT COUNT(*) FROM orders").fetchone()[0] == 3
assert con.execute("SELECT COUNT(*) FROM order_items").fetchone()[0] == 4
print("All modeling invariants passed.")
con.close()